In [2]:
import operator
import os
from typing import Annotated

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing_extensions import Literal

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import InjectedToolCallId, tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import InjectedState, ToolNode
from langgraph.types import Command
from langgraph.checkpoint.sqlite import SqliteSaver

In [3]:
load_dotenv()

OPENAI_API_KEY = os.environ['OPENAI_API_KEY']
CHROMA_DIR = "./chroma_db"

In [4]:
class RAGState(MessagesState):
    query: str
    retrieved_docs: Annotated[list[Document], operator.add]
    answer: str | None

In [5]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key = OPENAI_API_KEY
)

vectorstore = Chroma(
    persist_directory=CHROMA_DIR,
    collection_name="expert_interviews",
    embedding_function=embeddings,
)

In [6]:
class RetrieverInput(BaseModel):
    query: str = Field(description="Semantic query to search research paper chunks")
    market: Literal["France", "United Kingdom", "Germany"] = Field(description="The single market to retrieve from.")

@tool(args_schema=RetrieverInput)
def retrieve_market_docs(
    query: str,
    market: str,
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> Command:
    """
    Retrieve the top 3 relevant document chunks for ONE market.
    """

    docs = vectorstore.similarity_search(
        query,
        k=3,
        filter={"market": market},
    )

    summary = f"Retrieved {len(docs)} chunk(s) from the {market} market."

    return Command(
        update={
            "retrieved_docs": docs,
            "messages": [
                ToolMessage(
                    content=summary,
                    tool_call_id=tool_call_id,
                )
            ],
        }
    )

model = ChatOpenAI(model='gpt-5-mini', openai_api_key=OPENAI_API_KEY)
retrieval_model = model.bind_tools([retrieve_market_docs])

In [12]:
def setter_node(state: RAGState) -> dict:
    query = state['messages'][-1]
    return {'query': query}


RETRIEVAL_PROMPT = """
You are a research assistant capable of retrieving context. 
You have a retriever tool at your diposal which retrieves documents given a query and a market.
There are three markets - 'France', 'United Kingdom' and 'Germany'.
Retrieve important documents in regards to the user query.
You can only retrieve documents from one market in one call.
If multiple markets are present in the user query, you will need to make multiple calls.
If there is no market / country mentioned in the user's query, assume that the query is related to all the markets, in that case, retrieve relevant docs from all the markets. 
You DO NOT need to answer the user's query, just retrieve the relevant context.
When context is retrieved, you will only see a retrieval summary.
DO NOT PRODUCE the FINAL ANSWER, ONLY CALL TOOLS TO COLLECT CONTEXT."""

RETRIEVAL_SYSTEM_PROMPT = SystemMessage(content=RETRIEVAL_PROMPT)

def agent_node(state: RAGState) -> dict:
    messages = [RETRIEVAL_SYSTEM_PROMPT] + state['messages']
    response = retrieval_model.invoke(messages)
    updates: dict = {'messages': [response]}
    return updates


ANSWER_SYSTEM_PROMPT = SystemMessage(
    content="""
You are an answer generation assistant.

Answer the user's question using ONLY the retrieved research chunks.

MARKET -> 'France', 'Germany', 'United Kingdom'

If the user's query mentions a market, write only about that market. 
If two markets are mentioned, then write about both.
IF NO MARKET IS MENTIONED, WRITE ABOUT ALL THREE MARKETS.

For every factual claim, preserve its source and timestamp.

Citations MUST use this format:

[Source: <source> | Timestamp: <timestamp>]

When directly quoting a retrieved chunk, use quotation marks and include
the source and timestamp immediately after the quote.

Do not invent, modify, or omit source metadata.
Do not attribute information to a source unless that source actually
supports the claim.

If the retrieved information is insufficient, say so clearly.
Do not use outside knowledge. Do NOT ADD any knowledge outside from the context provided.

Write a very short and crisp answer. 
"""
)


def answer_node(state: RAGState) -> dict:
    docs = state.get("retrieved_docs", [])

    context_parts = []

    for i, doc in enumerate(docs):
        source = doc.metadata.get("source", "Unknown source")
        timestamp = doc.metadata.get("timestamp", "Unknown timestamp")
        market = doc.metadata.get("market", "Unknown market")

        context_parts.append(
            f"""
[Chunk {i + 1}]
Source: {source}
Timestamp: {timestamp}
Market: {market}

Content:
{doc.page_content}
"""
        )

    context = "\n".join(context_parts)

    prompt = HumanMessage(
        content=f"""
Question:
{state["query"]}

Retrieved research:

{context}

Answer the question using only the retrieved research.
Preserve source attribution and timestamps for every factual claim.
Select only relevant chunks that DIRECTLY answer the question. Ignore all irrelevant or merely related information. 
If there are no relevant chunks, simply return the message "No related data found in database"
"""
    )

    response = model.invoke([
        ANSWER_SYSTEM_PROMPT,
        prompt,
    ])

    return {
        "messages": [response],
        "answer": response.content
    }

def hallucination_checker(state):
    prompt = f"""
You are a hallucination removal step.

User query:
{state["query"]}

Retrieved context:
{state["retrieved_docs"]}

Generated answer:
{state["answer"]}

Rewrite the answer by removing any claim that is not directly supported
by the retrieved context.

Rules:
- Do not add new information.
- Do not use outside knowledge.
- Keep all claims that are supported.
- Preserve the original answer's structure and wording where possible.
- If a statement cannot be verified from the context, remove it.
- If the entire answer is unsupported, say that the retrieved context
  does not contain enough information to answer.

Return only the cleaned answer.
"""

    response = model.invoke(prompt)

    return {"answer": response.content}

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode


# Only tool available to the retrieval agent
tools = [retrieve_market_docs]

tool_node = ToolNode(tools)

def should_continue(state: RAGState):
    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tools"

    return "answer"


# Build graph
builder = StateGraph(RAGState)


builder.add_node("answer", answer_node)
builder.add_node("setter_node", setter_node)

builder.add_node("hallucination_checker", hallucination_checker)

builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "setter_node")
builder.add_edge("setter_node", 'agent')

builder.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        "answer": "answer",
    },
)

builder.add_edge("tools", "agent")
builder.add_edge("answer", 'hallucination_checker')
builder.add_edge('hallucination_checker', END)

import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

conn = sqlite3.connect(
    "checkpoints.db",
    check_same_thread=False
)

memory = SqliteSaver(conn)

graph = builder.compile(checkpointer=memory)

In [ ]:
test1b

{'messages': [HumanMessage(content='What are the roi considerations in france?', additional_kwargs={}, response_metadata={}, id='e8967346-6c6c-4b29-87ac-d7bf44f4bd79'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 500, 'prompt_tokens': 344, 'total_tokens': 844, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQzSumeO7X12S666d1JhQI2khdPr6', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0ca5d-452d-7c71-bb71-b709e567828b-0', tool_calls=[{'name': 'retrieve_market_docs', 'args': {'query': "ROI considerations in France:

In [8]:
config = {
    "configurable": {
        "thread_id": "test"
    }
}

result = graph.invoke(
    {"messages": HumanMessage(content='What are the roi considerations in france?')},
    config=config
)

In [9]:
result

{'messages': [HumanMessage(content='What are the roi considerations in france?', additional_kwargs={}, response_metadata={}, id='8b30724a-4ab5-4045-bcb3-348be4683520'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 202, 'prompt_tokens': 344, 'total_tokens': 546, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQzuC1F18LzmkXcSAqKZhx7zCXD3k', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0ca77-1900-7b51-80e8-d55b08bf6e07-0', tool_calls=[{'name': 'retrieve_market_docs', 'args': {'query': 'ROI considerations in France:

In [10]:
result2 = graph.invoke(
    {"messages": HumanMessage(content='answer the same question for germany')},
    config=config
)

In [11]:
result2

{'messages': [HumanMessage(content='What are the roi considerations in france?', additional_kwargs={}, response_metadata={}, id='8b30724a-4ab5-4045-bcb3-348be4683520'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 202, 'prompt_tokens': 344, 'total_tokens': 546, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQzuC1F18LzmkXcSAqKZhx7zCXD3k', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0ca77-1900-7b51-80e8-d55b08bf6e07-0', tool_calls=[{'name': 'retrieve_market_docs', 'args': {'query': 'ROI considerations in France: